Se requiere obtener información sobre el `total del presupuesto` y `total de ingresos` de las `películas` donde el `año de la fecha de lanzamiento` debe ser mayor o igual a 2015, también debe estar agrupado por el `año de la fecha de lanzamiento` y el `género` al que pertenece cada película.
También se requiere realizar un ranking ordenado de manera ascendente por el `total del presupuesto` y `total de ingresos` particionado por el `Año de la fecha de lanzamiento`

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

## 1. Obtenemos las películas y campos que nos interesan (`año de lanzamiento`, `budget`, `revenue`)

In [0]:
movies_df = spark.read.table("movie_silver.movies").filter(f"file_date = '{v_file_date}'")
movies_filtered_df = (movies_df
    .filter(movies_df.year_release_date >= 2015)
    .select("movie_id", "year_release_date", "budget", "revenue")
)
display(movies_filtered_df)


movie_id,year_release_date,budget,revenue


## 2. Obtener el `género` de la película

In [0]:
movies_genres_df = spark.read.table("movie_silver.movies_genres").filter(f"file_date = '{v_file_date}'")
genres_df = spark.read.table("movie_silver.genres")
movies_genres_name_df = (
    movies_genres_df.join(genres_df, on="genre_id", how="inner")
    .select("movie_id", "genre_name")
)
display(movies_genres_name_df)

movie_id,genre_name
5,Comedy
5,Crime
11,Adventure
11,Action
11,Science Fiction
12,Animation
12,Family
13,Drama
13,Comedy
13,Romance


## 3. Añadir `género` al DataFrame

In [0]:
movies_final_df = (
    movies_filtered_df.join(movies_genres_name_df, on="movie_id", how="inner")
    .select("year_release_date", "budget", "revenue", "genre_name")
)

display(movies_final_df)

year_release_date,budget,revenue,genre_name


## 4. Agrupar por el `año de la fecha de lanzamiento` y el `género`

In [0]:
from pyspark.sql.functions import sum
results_group_by_df = ( movies_final_df
                       .groupBy("year_release_date", "genre_name")
                       .agg(
                           sum("budget").alias("total_budget"),
                           sum("revenue").alias("total_revenue"),
                       )
)

## 5. Crear ranking ordenado de manera ascendente por el `total del presupuesto` y `total de ingresos` particionado por el `Año de la fecha de lanzamiento`

In [0]:
from pyspark.sql.functions import dense_rank, desc, lit
from pyspark.sql.window import Window

results_dense_rank_df = Window.partitionBy("year_release_date").orderBy(desc("total_budget"), desc("total_revenue"))
results_rank_df = results_group_by_df.withColumn("rank", dense_rank().over(results_dense_rank_df)).withColumn("created_date", lit(v_file_date))


display(results_rank_df)

year_release_date,genre_name,total_budget,total_revenue,rank,created_date


## 6. Escribir datos en el DataLake en formato `Delta`

In [0]:
merge_delta_lake( results_rank_df, "movie_gold", "results_group_movie_genre", "tgt.year_release_date = src.year_release_date AND tgt.genre_name = src.genre_name AND tgt.created_date = src.created_date", "created_date" )

In [0]:
%sql
SELECT * FROM movie_gold.results_group_movie_genre

year_release_date,genre_name,total_budget,total_revenue,rank,created_date
2015,Adventure,2.219E9,8.016412217E9,1,2024-12-30
2015,Action,2.0054E9,7.627777397E9,2,2024-12-30
2015,Drama,1.9518E9,6.473933286E9,3,2024-12-30
2015,Comedy,1.6232E9,6.131404237E9,4,2024-12-30
2015,Thriller,1.42115E9,5.617490431E9,5,2024-12-30
2015,Science Fiction,1.213425E9,4.333796155E9,6,2024-12-30
2015,Family,1.05635E9,4.235493125E9,7,2024-12-30
2015,Crime,9.268E8,1.961292814E9,8,2024-12-30
2015,Animation,7.21E8,3.532697782E9,9,2024-12-30
2015,Fantasy,5.76E8,1.415093823E9,10,2024-12-30
